In [1]:
import pandas as pd
from pathlib import Path

# =============================================================================
# PATHS
# =============================================================================

DATA_DIR = Path(
    "/home/root_1/Documents/CDL/repos/IDS-DRR_Heat/Heat-odisha/RiskScoreModel/data"
)
DEPLOY_DIR = DATA_DIR / "deployment"
DEPLOY_DIR.mkdir(parents=True, exist_ok=True)

DATA_DICTIONARY_CSV = DATA_DIR / "data_dictionary.csv"
DISTRICT_RISK_CSV = DATA_DIR / "district_final_risk_score.csv"
RC_RISK_CSV = DATA_DIR / "archive" / "final_risk_score.csv"

ID_COLUMNS = ["district", "dtname", "object-id", "object_id", "timeperiod"]

data_dict = pd.read_csv(DATA_DICTIONARY_CSV)
district_risk = pd.read_csv(DISTRICT_RISK_CSV)
rc_risk = pd.read_csv(RC_RISK_CSV)

indicator_slugs = set(data_dict["indicatorSlug"].dropna())

# =============================================================================
# COMPARE OUTPUT FILES AGAINST THE DATA DICTIONARY
# =============================================================================

def compare_against_dictionary(df, label):
    csv_cols = set(df.columns)
    undocumented = sorted(csv_cols - indicator_slugs - set(ID_COLUMNS))
    missing = sorted(indicator_slugs - csv_cols)

    print(f"\n--- {label} vs data_dictionary.csv ---")
    print(f"Undocumented columns in file ({len(undocumented)}): {undocumented}")
    print(f"Documented indicators missing from file ({len(missing)}): {missing}")

    return undocumented, missing


district_undocumented, district_missing = compare_against_dictionary(
    district_risk, "district_final_risk_score.csv"
)
rc_undocumented, rc_missing = compare_against_dictionary(
    rc_risk, "final_risk_score.csv"
)

with open(DEPLOY_DIR / "unmatched_columns.txt", "w") as f:
    f.write("district_final_risk_score.csv\n")
    f.write(f"  undocumented columns: {district_undocumented}\n")
    f.write(f"  missing documented indicators: {district_missing}\n\n")
    f.write("final_risk_score.csv\n")
    f.write(f"  undocumented columns: {rc_undocumented}\n")
    f.write(f"  missing documented indicators: {rc_missing}\n")

# =============================================================================
# DOCUMENTED DISTRICT-LEVEL AGGREGATION METHOD (for reference / manual review
# against the aggregation actually used in final_risk_score.ipynb)
# =============================================================================

agg_spec = data_dict.dropna(subset=["indicatorSlug"])[
    ["indicatorSlug", "District Level Aggregation", "cumulativeIndicator"]
]
print("\n--- Documented district-level aggregation spec ---")
print(agg_spec.to_string(index=False))

# =============================================================================
# DATA QUALITY CHECKS
# =============================================================================

print("\n--- Null counts (district_final_risk_score.csv) ---")
null_counts = district_risk.isna().sum()
print(null_counts[null_counts > 0])

fully_null_cols = district_risk.columns[district_risk.isna().all()].tolist()
if fully_null_cols:
    print(f"WARNING: fully-null columns: {fully_null_cols}")

duplicate_keys = district_risk.duplicated(
    subset=[c for c in ["district", "timeperiod"] if c in district_risk.columns]
).sum()
print(f"\nDuplicate (district, timeperiod) rows: {duplicate_keys}")

# =============================================================================
# BUILD DEPLOYMENT-READY FILES
# (identifier columns + documented model indicators + their cumulative
# companions, in a stable column order)
# =============================================================================

def build_deployment_file(df, out_name):
    cumulative_cols = set(data_dict["cumulativeIndicator"].dropna())
    deploy_cols = [c for c in ID_COLUMNS if c in df.columns] + [
        c for c in df.columns
        if c in indicator_slugs or c in cumulative_cols
    ]
    deploy_cols = list(dict.fromkeys(deploy_cols))  # de-dupe, preserve order

    deploy_df = df[deploy_cols].copy()
    out_path = DEPLOY_DIR / out_name
    deploy_df.to_csv(out_path, index=False)

    print(f"\nSaved deployment file: {out_path}")
    print(f"Rows: {len(deploy_df)}, Columns: {len(deploy_df.columns)}")

    return deploy_df


district_deploy = build_deployment_file(
    district_risk, "district_final_risk_score_deployment.csv"
)
rc_deploy = build_deployment_file(
    rc_risk, "final_risk_score_deployment.csv"
)


--- district_final_risk_score.csv vs data_dictionary.csv ---
Undocumented columns in file (7): ['land-surface-temperature-raster', 'men-bp', 'men-sugar', 'topsis-score', 'total-hhd', 'women-bp', 'women-sugar']
Documented indicators missing from file (0): []

--- final_risk_score.csv vs data_dictionary.csv ---
Undocumented columns in file (18): ['HealthCenters', 'avg-tele', 'block-area', 'block-name', 'block-nosanitation-hhds-pct', 'block-piped-hhds-pct', 'cum-tender-value', 'land-surface-temperature-raster', 'mean-heatday', 'men-bp', 'men-sugar', 'nco-5-9-percent-estimated', 'net-sown-area-in-hac', 'topsis-score', 'total-hhd', 'women-bp', 'women-sugar', 'year']
Documented indicators missing from file (6): ['health-centres-count', 'heat-days-score', 'nosanitation-hhds-pct', 'piped-hhds-pct', 'total-tender-awarded-value-fy-cumsum', 'workers-affected-pct']

--- Documented district-level aggregation spec ---
                       indicatorSlug District Level Aggregation                  